In [ ]:
import os
import zipfile
import scipy.io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import data_table
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score # Import confusion_matrix and additional metrics
import seaborn as sns
from scipy.io import loadmat
from scipy.stats import kurtosis, skew

In [ ]:
import random
import os

# Set a fixed seed value for reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# ==========================================
# 1. CONFIGURAÇÃO DAS CREDENCIAIS (DIRETO)
# ==========================================
# Acesse seu Kaggle > Settings > API > Create New Token
# Copie os valores do texto que aparecer e cole abaixo:

os.environ['KAGGLE_USERNAME'] = ""
os.environ['KAGGLE_KEY'] = ""


In [ ]:
# ==========================================
# 2. DOWNLOAD E EXTRAÇÃO DO DATASET CWRU
# ==========================================
print("--- INICIANDO DOWNLOAD DO DATASET CWRU ---")

# Baixa o dataset do Kaggle (link direto via API)
!kaggle datasets download -d brjapon/cwru-bearing-datasets

# Define os caminhos
zip_path = "cwru-bearing-datasets.zip"
extract_path = "cwru_data"

# Verifica se baixou e descompacta
if os.path.exists(zip_path):
    print("Extraindo arquivos...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    # Remove o zip para limpar o storage do Colab
    os.remove(zip_path)
    print(f"\n✅ SUCESSO: Dados extraídos em '/content/{extract_path}'")

    # Listar as pastas para confirmar
    print("\nPastas disponíveis:")
    !ls {extract_path}
else:
    print("\n❌ ERRO: O download não foi concluído. Verifique seu Username e Key.")

--- INICIANDO DOWNLOAD DO DATASET CWRU ---
Dataset URL: https://www.kaggle.com/datasets/brjapon/cwru-bearing-datasets
License(s): CC-BY-SA-4.0
100% 40.4M/40.4M [00:03<00:00, 13.4MB/s]

Extraindo arquivos...

✅ SUCESSO: Dados extraídos em '/content/cwru_data'

Pastas disponíveis:
CWRU_48k_load_1_CNN_data.npz  feature_time_48k_2048_load_1.csv	raw


In [ ]:
# ==========================================
# 1. CONFIGURAÇÕES E CAMINHOS
# ==========================================
input_folder = '/content/cwru_data/raw'
window_size = 2048  # Tamanho da janela conforme o seu projeto

# Lista dos teus ficheiros (Carga 1 - 48k)
files = [
    'B007_1_123.mat', 'B014_1_190.mat', 'B021_1_227.mat',
    'IR007_1_110.mat', 'IR014_1_175.mat', 'IR021_1_214.mat',
    'OR007_6_1_136.mat', 'OR014_6_1_202.mat', 'OR021_6_1_239.mat',
    'Time_Normal_1_098.mat'
]

raw_samples = []
labels = []

print("--- Iniciando Extração de Dados Brutos ---")

# ==========================================
# 2. CARREGAMENTO E SEGMENTAÇÃO (JANELAMENTO)
# ==========================================
for filename in files:
    path = os.path.join(input_folder, filename)

    if not os.path.exists(path):
        print(f"Aviso: {filename} não encontrado.")
        continue

    mat = loadmat(path)

    # Identifica a chave do sinal Drive End (DE)
    key = [k for k in mat.keys() if 'DE_time' in k][0]
    signal = mat[key].flatten()

    # Fatiar em janelas de 2048 pontos
    n_windows = len(signal) // window_size
    for i in range(n_windows):
        window = signal[i*window_size : (i+1)*window_size]
        raw_samples.append(window)
        # O rótulo ajuda a identificar a falha no espaço latente depois
        labels.append(filename.replace('.mat', ''))

X_raw = np.array(raw_samples)
y_labels = np.array(labels)

print(f"Total de janelas extraídas: {len(X_raw)}")

# ==========================================
# 4. DIVISÃO TREINO E TESTE (ESTRATIFICADA)
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_labels
)

print("\n--- Preparação Concluída ---")
print(f"Shape final Treino: {X_train.shape}")
print(f"Shape final Teste: {X_test.shape}")
print(f"Exemplo de Rótulo: {y_train[0]}")

In [ ]:
# 1. Normalização (Achata para 2D para o scaler funcionar)
scaler = StandardScaler()
X_train_flat = scaler.fit_transform(X_train)
X_test_flat = scaler.transform(X_test)

# 2. Adicionar a 3ª dimensão para a Conv1D
# Transforma (N, 2048) em (N, 2048, 1)
X_train_norm = X_train_flat.reshape((X_train_flat.shape[0], X_train_flat.shape[1], 1))
X_test_norm = X_test_flat.reshape((X_test_flat.shape[0], X_test_flat.shape[1], 1))

print(f"Novo shape de treino: {X_train_norm.shape}")
# Deve exibir: (Amostras, 2048, 1)

Novo shape de treino: (1895, 2048, 1)


In [ ]:
# ==========================================
# 8. TRANSFORMANDO A SAÍDA PARA VALORES NUMÉRICOS
# ==========================================
# 1. Transformar as strings (Normal, Ball, etc.) em números (0, 1, 2...)
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

labels = le.classes_

**####################################################################################**

In [ ]:
# --- 1. DEFINIR APENAS O EXTRATOR ---
# Parâmetros baseados no seu arquivo
input_shape = X_train_norm.shape[1:] # (2048, 1)
num_classes = len(np.unique(y_train))

def build_pure_vit_optimized():
    inputs = layers.Input(shape=input_shape)

    # 1. Pré-processamento e Ruído (Regularização)
    x = layers.GaussianNoise(0.01)(inputs)

    # 2. Patch Embedding com Sobreposição
    # kernel=64 e stride=32 cria 50% de sobreposição entre patches
    x = layers.Conv1D(64, kernel_size=64, strides=32, padding='same')(x)
    x = layers.LayerNormalization()(x)

    # 3. Position Embedding (Ajustado para o novo número de patches)
    num_patches = x.shape[1]
    pos_emb = tf.Variable(initial_value=tf.random.normal(shape=(1, num_patches, 64), stddev=0.01))
    x = x + pos_emb

    # --- BLOCO TRANSFORMER ENCODER ---
    # Sub-bloco 1: Self-Attention + Residual
    attn_out = layers.MultiHeadAttention(num_heads=8, key_dim=64)(x, x)
    x1 = layers.Add()([x, attn_out]) # Conexão Residual
    x1 = layers.LayerNormalization()(x1)

    # Sub-bloco 2: Feed-Forward (MLP) + Residual
    ffn = layers.Dense(128, activation='gelu')(x1)
    ffn = layers.Dense(64)(ffn)
    x2 = layers.Add()([x1, ffn]) # Conexão Residual
    x2 = layers.LayerNormalization()(x2)

    # 4. Classificação Final
    x = layers.GlobalAveragePooling1D()(x2)
    features = layers.Dense(256, activation='gelu')(x)

    extractor_model = keras.Model(inputs, features, name="ViT_Extractor_Puro")
    return extractor_model

In [ ]:
# --- 2. CRIAR O OBJETO DO EXTRATOR ---
vit_extractor = build_pure_vit_optimized()

# --- 3. ADICIONAR A CLASSIFICAÇÃO APENAS PARA TREINAR ---
# Criamos um "modelo de treino" que consome a saída do extrator
training_inputs = layers.Input(shape=input_shape)
extracted_features = vit_extractor(training_inputs) # O extrator é usado aqui
classification_head = layers.Dense(num_classes, activation='softmax')(extracted_features)

vit_for_training = keras.Model(training_inputs, classification_head)

In [ ]:
# --- 4. TREINAR ---
vit_for_training.compile(
    optimizer=keras.optimizers.Adam(0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callback para evitar overfitting
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

print("Iniciando o treinamento do ViT para aprender as assinaturas de falha...")
history = vit_for_training.fit(
    X_train_norm, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stopping],
    verbose=1
)

Iniciando o treinamento do ViT para aprender as assinaturas de falha...
Epoch 1/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 27s 250ms/step - accuracy: 0.5437 - loss: 1.4981 - val_accuracy: 0.9211 - val_loss: 0.6612
Epoch 2/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 16s 170ms/step - accuracy: 0.9537 - loss: 0.3275 - val_accuracy: 0.9632 - val_loss: 0.1515
Epoch 3/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 8s 151ms/step - accuracy: 0.9806 - loss: 0.0962 - val_accuracy: 0.9895 - val_loss: 0.0707
Epoch 4/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 8s 145ms/step - accuracy: 0.9871 - loss: 0.0512 - val_accuracy: 0.9632 - val_loss: 0.1155
Epoch 5/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 9s 169ms/step - accuracy: 0.9842 - loss: 0.0565 - val_accuracy: 0.9895 - val_loss: 0.0414
Epoch 6/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 7s 130ms/step - accuracy: 0.9912 - loss: 0.0279 - val_accuracy: 0.9895 - val_loss: 0.0288
Epoch 7/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 12s 159ms/step - accuracy: 0.9912 - loss: 0.0272 - val_accuracy: 0.9895 - val_loss: 0.0311
Epoch 8/50
54/54 ━━━━━━━━━━━━

In [ ]:
# --- 5. EXPORTAR ---
vit_extractor.save("extrator_VIT.keras")
print("Modelo salvo com sucesso!")

Modelo salvo com sucesso!
